In [6]:
# Autoencoder for Image Denoising using PyTorch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Optional: torchsummary
try:
    from torchsummary import summary
    HAS_SUMMARY = True
except ImportError:
    HAS_SUMMARY = False

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


In [7]:
# Device configuration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Running on CPU.")

Using device: cpu
CUDA is not available. Running on CPU.


In [8]:
transform = transforms.Compose([
    transforms.ToTensor()
])

In [9]:
# Load MNIST dataset

dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

print("Training samples:", len(dataset))
print("Testing samples:", len(test_dataset))

Training samples: 60000
Testing samples: 10000


In [10]:
# Add noise to images

def add_noise(inputs, noise_factor=0.5):
    noisy = inputs + noise_factor * torch.rand_like(inputs)
    return torch.clamp(noisy, 0., 1.)

In [12]:
# Define Denoising Autoencoder

class DenoisingAutoencoder(nn.Module):

    def __init__(self):
        super(DenoisingAutoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                64, 32,
                kernel_size=2,
                stride=2
            ),
            nn.ReLU(),

            nn.ConvTranspose2d(
                32, 1,
                kernel_size=2,
                stride=2
            ),
            nn.Sigmoid()
        )

    def forward(self, x):

        encoded = self.encoder(x)

        decoded = self.decoder(encoded)

        return decoded

In [13]:
# Initialize model, loss function and optimizer

model = DenoisingAutoencoder().to(device)

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

DenoisingAutoencoder(
  (encoder): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (decoder): Sequential(
    (0): ConvTranspose2d(64, 32, kernel_size=(2, 2), stride=(2, 2))
    (1): ReLU()
    (2): ConvTranspose2d(32, 1, kernel_size=(2, 2), stride=(2, 2))
    (3): Sigmoid()
  )
)


In [14]:
# Print model summary

if HAS_SUMMARY:
    summary(model, input_size=(1, 28, 28))
else:
    print("torchsummary is not installed.")
    print(model)

total_params = sum(p.numel() for p in model.parameters())

print("Total parameters:", total_params)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 28, 28]             320
              ReLU-2           [-1, 32, 28, 28]               0
         MaxPool2d-3           [-1, 32, 14, 14]               0
            Conv2d-4           [-1, 64, 14, 14]          18,496
              ReLU-5           [-1, 64, 14, 14]               0
         MaxPool2d-6             [-1, 64, 7, 7]               0
   ConvTranspose2d-7           [-1, 32, 14, 14]           8,224
              ReLU-8           [-1, 32, 14, 14]               0
   ConvTranspose2d-9            [-1, 1, 28, 28]             129
          Sigmoid-10            [-1, 1, 28, 28]               0
Total params: 27,169
Trainable params: 27,169
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.75
Params size (MB): 0.10
Estimated Tot

In [15]:
# Train the autoencoder

def train(model, loader, criterion, optimizer, epochs=5):

    model.train()

    losses = []

    for epoch in range(epochs):

        running_loss = 0.0

        for images, _ in loader:

            # Move images to CPU/GPU
            images = images.to(device)

            # Create noisy images
            noisy_images = add_noise(images)

            # Clear gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(noisy_images)

            # Calculate reconstruction loss
            loss = criterion(outputs, images)

            # Backpropagation
            loss.backward()

            # Update weights
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(loader)

        losses.append(epoch_loss)

        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"Loss: {epoch_loss:.4f}"
        )

    return losses

In [16]:
# Plot training loss

def plot_loss(losses):

    plt.figure(figsize=(8, 5))

    plt.plot(
        range(1, len(losses) + 1),
        losses,
        marker='o'
    )

    plt.title("Autoencoder Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.grid(True)

    plt.show()

In [17]:
# Visualize denoising results

def visualize_denoising(model, loader, num_images=10):

    model.eval()

    with torch.no_grad():

        for images, _ in loader:

            images = images.to(device)

            # Add noise
            noisy_images = add_noise(images)

            # Denoise
            outputs = model(noisy_images)

            break

    images = images.cpu().numpy()
    noisy_images = noisy_images.cpu().numpy()
    outputs = outputs.cpu().numpy()

    print("Name:")
    print("Register Number:")

    plt.figure(figsize=(15, 6))

    for i in range(num_images):

        # Original
        ax = plt.subplot(3, num_images, i + 1)
        plt.imshow(
            images[i].squeeze(),
            cmap="gray"
        )
        ax.set_title("Original")
        ax.axis("off")

        # Noisy
        ax = plt.subplot(
            3,
            num_images,
            i + 1 + num_images
        )

        plt.imshow(
            noisy_images[i].squeeze(),
            cmap="gray"
        )

        ax.set_title("Noisy")
        ax.axis("off")

        # Denoised
        ax = plt.subplot(
            3,
            num_images,
            i + 1 + 2 * num_images
        )

        plt.imshow(
            outputs[i].squeeze(),
            cmap="gray"
        )

        ax.set_title("Denoised")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Run training and visualization

losses = train(
    model,
    train_loader,
    criterion,
    optimizer,
    epochs=1
)

plot_loss(losses)

visualize_denoising(
    model,
    test_loader,
    num_images=10
)